# NeurIPS 2026 Smart Buildings — Submission Version 4

## New experiment: predictive supervisory HVAC control (not V2/V3 PID)

**Design:** A lightweight online zone-trend forecaster (15-minute horizon), an AHU-level heating/cooling/off state machine with start/stop hysteresis, 30-minute pre-occupancy conditioning, direct supply-air temperature resets, per-zone damper/reheat coordination, and a central water-demand governor.

**Plant constraints:** Air handlers **only cool**; heating is delivered by hot-water-fed VAV reheat coils. A high AHU supply setpoint avoids artificial cooling in heating mode. Continuous actions use normalized `[-1,1]`; `+1` recirculation requests the simulator's capped maximum (70%). Live metadata supplies actuator/observation mappings. Uses CPU/NumPy only; no external files, training, weather labels, or model downloads.

### What this tests

- **V2:** mean-zone AHU PID + proportional dampers/reheat + economizer/water reset; public loss **0.06701**.
- **V3:** modifies V2's PID with guarded outlier correction and demand-gated hot water; public loss **0.06207**.
- **V4:** **no PID**. Predicts short-horizon zone excursions, operates only the needed AHUs, selects cooling vs reheat-dominant operation, and uses explicit hysteresis; designed as an architectural ablation rather than a V3 patch.

The notebook is **not yet scored** on the full training simulator or the Kaggle leaderboard. Passing local unit/synthetic tests is only an interface and control-logic check, not evidence of score improvement.

### Ablation switches

`USE_TREND`, `USE_HYSTERESIS`, and `USE_PRECONDITION` can be independently disabled for a controlled experiment. Compare **full 8,640-step episodes** over all 30 shipped training scenarios with the same weather/building seeds and report comfort, energy, and carbon separately. Keep V3 as reference. Set expected lower-is-better results only after official evaluation.

**Submission:** Kaggle's inference gateway runs the rollout; no manual `submission.parquet`. CPU, internet off, no install step. Sources: [official starter kit](https://github.com/smart-buildings-2026/starter-kit), [policy contract](https://github.com/smart-buildings-2026/starter-kit/blob/main/smart_buildings/baselines/transfer_policy.py).


In [ ]:
"""V4: predictive, demand-led supervisory HVAC control (no PID)."""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Generalizes via live metadata; no saved models, climate IDs, or tuning data."""

    # Ablations are intentionally independent of the V2/V3 PID design.
    USE_TREND = True
    USE_HYSTERESIS = True
    USE_PRECONDITION = True
    PREDICTION_STEPS = 3  # 15 minutes at five-minute sampling
    START_K = 0.22
    STOP_K = 0.12
    MIN_RUN_STEPS = 3
    PRESTART_HOURS = 0.5

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.n_cont = int(m['n_continuous_action'])
        self.n_disc = int(m['n_discrete_action'])
        self.temp_idx = list(map(int, m['temp_indices']))
        self.press_idx = list(map(int, m['pressure_indices']))
        self.recirc_idx = list(map(int, m['recirculation_indices']))
        self.water_idx = list(map(int, m['water_indices']))
        self.n_ahu = len(self.temp_idx)
        self.water_pressure_idx = (self.press_idx[self.n_ahu]
                                   if len(self.press_idx) > self.n_ahu else None)
        self.water_temperature_idx = self.water_idx[0] if self.water_idx else None
        self.water_run_idx = self.n_disc - 1
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        norm = m.get('observation_normalization_metadata', {}).get(
            'zone_air_temperature_sensor', {})
        self.zone_mean = float(norm.get('sample_mean', 0.0))
        self.zone_std = float(norm.get('sample_std', 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        assignments = m['ahu_zone_assignments']
        action_map = m['zone_action_indices']

        def ahu_sort_key(name):
            match = re.search(r'\d+', str(name))
            return (int(match.group()) if match else -1, str(name))

        self.ahus = []
        self.zone_ids = []
        for i, ahu_id in enumerate(sorted(assignments, key=ahu_sort_key)[:self.n_ahu]):
            zones = []
            for room in assignments[ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                zone_index = self.fields.get(vav + '_zone_air_temperature_sensor')
                if zone_index is None:
                    continue
                actions = action_map.get(vav, {})
                zones.append({
                    'id': vav,
                    'temp': zone_index,
                    'lo': self.fields.get(vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'hi': self.fields.get(vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper': actions.get('supply_air_damper_percentage_command'),
                    'reheat': actions.get('reheat_valve_percentage_command'),
                })
                self.zone_ids.append(vav)
            self.ahus.append({
                'zones': zones,
                'temp': self.temp_idx[i],
                'pressure': self.press_idx[i] if i < len(self.press_idx) else None,
                'recirc': self.recirc_idx[i] if i < len(self.recirc_idx) else None,
                'outside': self.fields.get(f'ahs_{ahu_id}_outside_air_temperature_sensor'),
                'run': i,
            })
        self.reset()

    def reset(self):
        self.prev_temp = {}
        self.trend = {}
        self.running = [False] * len(self.ahus)
        self.run_steps = [0] * len(self.ahus)

    @staticmethod
    def _clamp(x, lo=0.0, hi=1.0):
        return float(np.clip(x, lo, hi))

    @staticmethod
    def _norm_fraction(x):
        return 2.0 * float(np.clip(x, 0.0, 1.0)) - 1.0

    def _schedule(self, obs):
        def cyclic(s, c, period):
            a, b = self.fields.get(s), self.fields.get(c)
            if a is None or b is None:
                return None
            x, y = float(obs[a]), float(obs[b])
            if not (np.isfinite(x) and np.isfinite(y)):
                return None
            return ((np.arctan2(x, y) % (2.0 * np.pi)) * period / (2.0 * np.pi))
        hour = cyclic('hod_sin_000', 'hod_cos_000', 24.0)
        day = cyclic('dow_sin_000', 'dow_cos_000', 7.0)
        if hour is None or day is None:
            return self.eco, False
        weekday = int(np.floor(day + 1e-6)) < 5
        occupied = weekday and self.morning <= hour < self.evening
        prestart = (self.USE_PRECONDITION and weekday and
                    self.morning - self.PRESTART_HOURS <= hour < self.morning)
        return self.comfort if occupied or prestart else self.eco, prestart

    def _read_zone(self, obs, zone, fallback):
        t = self.zone_mean + float(obs[zone['temp']]) * self.zone_std
        if not np.isfinite(t) or t < 240 or t > 340:
            return None
        low = float(obs[zone['lo']]) if zone['lo'] is not None else float('nan')
        high = float(obs[zone['hi']]) if zone['hi'] is not None else float('nan')
        if not (np.isfinite(low) and np.isfinite(high) and
                245 <= low < high <= 335 and high - low <= 35):
            low, high = fallback
        # The sensor represents the current scoring band. Only the intentional
        # 30-minute prestart uses the forthcoming comfort band instead.
        if self._prestart:
            low, high = fallback
        key = zone['id']
        previous = self.prev_temp.get(key)
        if previous is not None:
            delta = self._clamp(t - previous, -0.30, 0.30)
            self.trend[key] = 0.70 * self.trend.get(key, 0.0) + 0.30 * delta
        else:
            self.trend[key] = 0.0
        self.prev_temp[key] = t
        slope = self.trend[key] if self.USE_TREND else 0.0
        predicted = t + self.PREDICTION_STEPS * slope
        # Protect against deadband drift: do not lead more than 0.6 K.
        predicted = t + self._clamp(predicted - t, -0.6, 0.6)
        hot = max(0.0, predicted - high)
        cold = max(0.0, low - predicted)
        return {'zone': zone, 'temp': t, 'hot': hot, 'cold': cold,
                'low': low, 'high': high}

    def _choose_mode(self, i, hot, cold):
        demand = max(hot, cold)
        if not self.USE_HYSTERESIS:
            on = demand >= self.START_K
        elif not self.running[i]:
            on = demand >= self.START_K
        elif self.run_steps[i] < self.MIN_RUN_STEPS:
            on = True
        else:
            on = demand > self.STOP_K
        self.running[i] = bool(on)
        self.run_steps[i] = (self.run_steps[i] + 1) if on else 0
        if not on:
            return 'off'
        return 'cool' if hot >= cold and hot >= 0.08 else 'heat'

    def _recirculation(self, outdoor, indoor, target, pressure_pa):
        # Full recirculation command is physically clamped to 70% by simulator.
        if not np.isfinite(outdoor) or outdoor >= indoor:
            return 1.0
        if indoor - outdoor <= pressure_pa * 0.00322:
            return 1.0
        # Do not overdraw cold outside air: min intake fraction is 30%.
        fraction_outdoor = self._clamp((indoor - target) /
                                       max(1e-6, indoor - outdoor), 0.30, 1.0)
        return self._norm_fraction(1.0 - fraction_outdoor)

    def get_action(self, obs):
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        fallback, self._prestart = self._schedule(obs)
        continuous = np.full(self.n_cont, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_disc, dtype=np.int32)
        # AHUs only cool; a high setpoint asks for *no active cooling*.
        for j in self.temp_idx:
            continuous[j] = 1.0
        for j in self.recirc_idx:
            continuous[j] = 1.0

        water_shortfalls = []
        active_heat_count = 0
        for i, ahu in enumerate(self.ahus):
            zs = [record for zone in ahu['zones']
                  if (record := self._read_zone(obs, zone, fallback)) is not None]
            if not zs:
                self.running[i] = False
                self.run_steps[i] = 0
                continue
            hot = max(z['hot'] for z in zs)
            cold = max(z['cold'] for z in zs)
            mode = self._choose_mode(i, hot, cold)
            if mode == 'off':
                continue
            discrete[ahu['run']] = 1
            # Power control uses severity and *fraction of zones* in demand;
            # unlike PID, there is no integration or feedback on mean zone T.
            extent = max(hot, cold)
            n_served = sum(max(z['hot'], z['cold']) >= 0.15 for z in zs)
            coverage = max(0.30, n_served / max(1, len(zs)))
            pressure_pa = 200.0 + 1550.0 * self._clamp(extent / 3.0) * np.sqrt(coverage)
            pressure_pa = float(np.clip(pressure_pa, 200.0, 1750.0))
            if ahu['pressure'] is not None:
                continuous[ahu['pressure']] = self._clamp(
                    pressure_pa / 1000.0 - 1.0, -1.0, 1.0)
            if mode == 'cool':
                # Direct supply air temperature reset, not signed PID gain.
                target = np.clip(293.0 - 2.75 * hot, 285.0, 296.0)
                continuous[ahu['temp']] = self._clamp(
                    (target - 285.0) / 10.0 - 1.0, -1.0, 1.0)
                oi = ahu['outside']
                outdoor = float(obs[oi]) if oi is not None else float('nan')
                if ahu['recirc'] is not None:
                    indoor = float(np.mean([z['temp'] for z in zs]))
                    continuous[ahu['recirc']] = self._recirculation(
                        outdoor, indoor, float(target), pressure_pa)
            else:
                # No heating coil in AHU: leave cooling setpoint high.
                continuous[ahu['temp']] = 1.0
            for z in zs:
                zone = z['zone']
                # Opposite-demand zones normally stay at their ventilation floor,
                # but a cold zone can receive reheat while its AHU is cooling.
                if mode == 'cool':
                    airflow = max(self._clamp(z['hot'] / 2.5),
                                  0.30 * self._clamp(z['cold'] / 1.5))
                else:
                    airflow = self._clamp(z['cold'] / 2.5)
                if zone['damper'] is not None:
                    continuous[zone['damper']] = self._norm_fraction(airflow)
                if z['cold'] > 0.08 and zone['reheat'] is not None:
                    # The water system must run whenever a reheat valve is open.
                    valve = self._clamp(z['cold'] / 2.5)
                    continuous[zone['reheat']] = self._norm_fraction(valve)
                    water_shortfalls.append(z['cold'])
                    active_heat_count += 1

        if active_heat_count:
            # System-wide reset on high-percentile heating demand (robust to
            # a single extreme zone); water pressure scaled by valve demand.
            shortfall = float(np.percentile(water_shortfalls, 75))
            intensity = self._clamp(shortfall / 3.0)
            if self.water_temperature_idx is not None:
                # Native hot-water setpoint 310..350 K.
                water_k = 318.0 + 30.0 * intensity
                continuous[self.water_temperature_idx] = self._norm_fraction(
                    (water_k - 310.0) / 40.0)
            if self.water_pressure_idx is not None:
                continuous[self.water_pressure_idx] = self._norm_fraction(
                    max(0.45, intensity))
            if 0 <= self.water_run_idx < self.n_disc:
                discrete[self.water_run_idx] = 1

        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()